# ST1 — Critical Minerals: Data Ingestion & Cleaning
**Team 7 Lambda | Phase 1**

**Goal:** Pull, clean, and save three data sources:
1. BGS World Mineral Statistics — production by country (CSVs downloaded from BGS, placed in `data/raw/st1/`)
2. World Bank Pink Sheet — commodity prices (auto-downloaded)
3. OECD Export Restrictions — policy events by country

**Output:** `data/processed/st1_minerals_production.parquet`, `st1_hhi.parquet`, `st1_prices.parquet`, `st1_oecd_restrictions.parquet`

**BGS CSV format (long):** Each file has columns `country_iso3_code`, `country_trans`, `year` (datetime string), `quantity`, `units`, `bgs_commodity_trans`

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append('../../src')   # src/config.py and src/utils.py

import pandas as pd
import numpy as np
import requests
from pathlib import Path

from config import (DATA_RAW, DATA_PROC, FOCAL_COUNTRIES, START_YEAR, END_YEAR,
                    ST1_BGS, ST1_WORLDBANK, ST1_OECD, ST1_PROC)
from utils import compute_hhi, flag_concentration, save_parquet, set_style, log

set_style()
log.info('ST1 setup complete.')

18:19:42 [INFO] ST1 setup complete.


## 1A — BGS World Mineral Statistics (Production by Country)

**Source:** https://www.bgs.ac.uk/mineralsuk/statistics/world-mineral-statistics/  
**Files placed in:** `data/raw/st1/`  
**Format:** BGS long-format CSV — each row is one country × year observation.  
Key columns: `country_iso3_code`, `country_trans`, `year` (datetime string), `quantity`, `units`, `bgs_commodity_trans`

**Minerals available (11):** cobalt, copper, gallium, germanium, gold, graphite, lithium, manganese_ore, nickel, rare_earth, silver  
**Not available (no BGS data):** silicon

In [2]:
# ── BGS Directory & Mineral Map ─────────────────────────────────────────────
BGS_DIR = ST1_BGS   # data/raw/ST1/bgs/

# Map substrings in bgs_commodity_trans or filename → canonical mineral label
MINERAL_MAP = {
    'rare earth':      'rare_earths',
    'manganese ore':   'manganese',
    'manganese':       'manganese',
    'lithium mineral': 'lithium',
    'lithium':         'lithium',
    'cobalt':          'cobalt',
    'gallium':         'gallium',
    'germanium':       'germanium',
    'graphite':        'graphite',
    'nickel':          'nickel',
    'copper':          'copper',
    'silicon':         'silicon',
    'gold':            'gold',
    'silver':          'silver',
}

In [3]:
# ── BGS CSV Loader ──────────────────────────────────────────────────────────

def map_mineral(name: str) -> str | None:
    """
    Map a BGS commodity name or filename stem to the project canonical mineral label.
    Matches longest keyword first to avoid partial matches.

    Args:
        name: raw commodity name string (from bgs_commodity_trans or filename)

    Returns:
        Canonical mineral label string or None if no match.

    Usage:
        label = map_mineral('cobalt, mine')  # → 'cobalt'
    """
    if pd.isna(name):
        return None
    name_lower = str(name).strip().lower()
    for keyword in sorted(MINERAL_MAP.keys(), key=len, reverse=True):
        if keyword in name_lower:
            return MINERAL_MAP[keyword]
    return None


def load_bgs_csv(fpath: Path) -> pd.DataFrame:
    """
    Load a single BGS long-format CSV file.
    Expected columns: country_iso3_code, country_trans, year, quantity,
                      units, bgs_commodity_trans.

    Args:
        fpath: path to BGS CSV file

    Returns:
        Cleaned DataFrame with columns: mineral, iso3, country, year, production, units.
        Empty DataFrame if file cannot be parsed.

    Usage:
        df = load_bgs_csv(BGS_DIR / 'cobalt.csv')
    """
    try:
        df = pd.read_csv(fpath)
    except Exception as e:
        log.error(f'Cannot read {fpath.name}: {e}')
        return pd.DataFrame()

    df.columns = df.columns.str.strip()
    log.info(f'Reading {fpath.name}: {df.shape}')

    # ── Identify key columns ────────────────────────────────────────────────
    iso3_col    = next((c for c in df.columns if 'iso3' in c.lower()), None)
    country_col = next((c for c in df.columns
                        if 'country_trans' in c.lower() or
                        ('country' in c.lower() and 'iso' not in c.lower())), None)
    year_col    = next((c for c in df.columns if c.lower() == 'year'), None)
    qty_col     = next((c for c in df.columns if 'quantity' in c.lower()), None)
    units_col   = next((c for c in df.columns if 'units' in c.lower()), None)

    # Prefer bgs_commodity_trans exactly; fall back to other commodity cols
    # Important: avoid bgs_sub_commodity_trans which is often all-NaN
    commodity_col = None
    for c in df.columns:
        if c.lower() == 'bgs_commodity_trans':
            commodity_col = c
            break
    if commodity_col is None:
        for c in df.columns:
            if 'bgs_commodity' in c.lower() and 'sub' not in c.lower():
                commodity_col = c
                break

    if not all([iso3_col, year_col, qty_col]):
        log.warning(f'{fpath.name}: missing required columns '
                    f'(iso3={iso3_col}, year={year_col}, qty={qty_col}). Skipping.')
        return pd.DataFrame()

    # ── Parse year from datetime string (e.g. '2000-01-01 00:00:00') ────────
    df['year'] = pd.to_datetime(df[year_col], errors='coerce').dt.year
    df['year'] = df['year'].astype('Int64')

    # ── Quantity to numeric ──────────────────────────────────────────────────
    df['production'] = (
        df[qty_col].astype(str)
        .str.replace(',', '', regex=False)
        .str.replace(r'[^0-9.]', '', regex=True)
        .replace('', np.nan)
        .pipe(pd.to_numeric, errors='coerce')
    )

    # ── Determine mineral label ──────────────────────────────────────────────
    # Prefer bgs_commodity_trans column; fall back to filename stem
    if commodity_col and df[commodity_col].notna().any():
        df['mineral'] = df[commodity_col].map(map_mineral)
    else:
        log.warning(f'{fpath.name}: commodity column empty or missing — using filename stem.')
        df['mineral'] = map_mineral(fpath.stem)

    unmapped = df['mineral'].isna().sum()
    if unmapped:
        sample = df.loc[df['mineral'].isna(),
                        commodity_col if commodity_col else year_col].unique()[:5]
        log.warning(f'{fpath.name}: {unmapped} unmapped rows. Add to MINERAL_MAP if needed: {sample}')

    # ── Build output ─────────────────────────────────────────────────────────
    out = pd.DataFrame()
    out['mineral']    = df['mineral']
    out['iso3']       = df[iso3_col].astype(str).str.strip().str.upper()
    out['country']    = df[country_col].astype(str).str.strip() if country_col else out['iso3']
    out['year']       = df['year']
    out['production'] = df['production']
    out['units']      = df[units_col].astype(str).str.strip() if units_col else 'unknown'

    skip_iso = {'nan', '', 'none', 'world', 'total'}
    out = out[~out['iso3'].str.lower().isin(skip_iso)]
    out = out.dropna(subset=['mineral', 'iso3', 'year', 'production'])

    log.info(f'  {fpath.name}: {out.shape} rows | minerals: {out["mineral"].unique()}')
    return out


def load_all_bgs() -> pd.DataFrame:
    """
    Load all BGS CSV files from BGS_DIR, parse each, and concatenate.
    Aggregates sub-commodities (e.g. lithium + lithium minerals → 'lithium') by summing.

    Returns:
        Long-format DataFrame: mineral, iso3, country, year, production.
        Empty DataFrame if no files found.

    Usage:
        prod_df = load_all_bgs()
    """
    files = sorted(
        list(BGS_DIR.glob('*.csv')) +
        list(BGS_DIR.glob('*.xlsx')) +
        list(BGS_DIR.glob('*.xls'))
    )

    if not files:
        log.warning(f'No BGS files found in {BGS_DIR}.')
        return pd.DataFrame(columns=['mineral', 'iso3', 'country', 'year', 'production'])

    parts = [load_bgs_csv(f) for f in files]
    parts = [p for p in parts if not p.empty]

    if not parts:
        log.warning('No parseable BGS data found.')
        return pd.DataFrame(columns=['mineral', 'iso3', 'country', 'year', 'production'])

    combined = pd.concat(parts, ignore_index=True)
    combined = combined[combined['year'].between(START_YEAR, END_YEAR)]

    prod = (
        combined
        .groupby(['mineral', 'iso3', 'country', 'year'], as_index=False)['production']
        .sum(min_count=1)
    )

    log.info(f'BGS combined: {prod.shape} | minerals: {sorted(prod["mineral"].unique())}')
    return prod[['mineral', 'iso3', 'country', 'year', 'production']]

In [4]:
# ── Load BGS Production Data ─────────────────────────────────────────────────
prod_df = load_all_bgs()

if not prod_df.empty:
    log.info(f'Minerals loaded: {sorted(prod_df["mineral"].unique())}')
    log.info(f'Year range: {prod_df["year"].min()} – {prod_df["year"].max()}')
    display(prod_df.groupby('mineral')[['production']].describe().round(0))
else:
    log.warning('prod_df empty — check files in data/raw/st1/')

18:19:42 [WARNING] No BGS files found in /Users/taruntheegela/Desktop/lambda/data/raw/ST1/bgs.
18:19:42 [WARNING] prod_df empty — check files in data/raw/st1/


In [5]:
# ── Compute HHI per Mineral per Year ────────────────────────────────────────
# HHI = Σ(share²), range 0–1.  > 0.25 = highly concentrated (config.HHI_HIGH_THRESHOLD).

if not prod_df.empty:
    hhi_results = []
    for mineral, group in prod_df.groupby('mineral'):
        hhi_by_year = (
            group.dropna(subset=['production'])
            .groupby('year')
            .apply(lambda g: ((g['production'] / g['production'].sum()) ** 2).sum())
            .reset_index()
            .rename(columns={0: 'hhi'})
        )
        hhi_by_year['mineral'] = mineral
        hhi_results.append(hhi_by_year)

    hhi_df = pd.concat(hhi_results, ignore_index=True)
    hhi_df['high_concentration'] = flag_concentration(hhi_df['hhi'])

    log.info(f'HHI computed: {hhi_df.shape}')
    display(hhi_df.pivot(index='year', columns='mineral', values='hhi').round(3))
else:
    log.warning('prod_df empty — skipping HHI.')
    hhi_df = pd.DataFrame(columns=['year', 'mineral', 'hhi', 'high_concentration'])

18:19:42 [WARNING] prod_df empty — skipping HHI.


## 1B — World Bank Pink Sheet (Commodity Prices)

**Source:** https://thedocs.worldbank.org/en/doc/18675f1d1639c7a34d463f59263ba0a2-0050012025/related/CMO-Historical-Data-Monthly.xlsx  
**Auto-download:** Yes — runs automatically below. No manual step needed.  
**Coverage:** Monthly prices 1960–present for 70+ commodities including copper, nickel, cobalt, gold, silver.

In [6]:
# ── Auto-Download World Bank Pink Sheet ─────────────────────────────────────
WB_PATH = ST1_WORLDBANK / 'pink_sheet.xlsx'   # data/raw/ST1/worldbank/pink_sheet.xlsx

WB_URL = 'https://thedocs.worldbank.org/en/doc/18675f1d1639c7a34d463f59263ba0a2-0050012025/related/CMO-Historical-Data-Monthly.xlsx'

if not WB_PATH.exists():
    log.info('Downloading World Bank Pink Sheet...')
    try:
        r = requests.get(WB_URL, timeout=60)
        r.raise_for_status()
        WB_PATH.write_bytes(r.content)
        log.info(f'Saved Pink Sheet: {WB_PATH} ({len(r.content):,} bytes)')
    except Exception as e:
        log.error(
            f'Auto-download failed: {e}\n'
            f'Manual fallback: save pink_sheet.xlsx to {ST1_WORLDBANK}'
        )
else:
    log.info(f'Pink Sheet already exists: {WB_PATH}')

18:19:42 [INFO] Downloading World Bank Pink Sheet...
18:19:42 [INFO] Saved Pink Sheet: /Users/taruntheegela/Desktop/lambda/data/raw/ST1/worldbank/pink_sheet.xlsx (778,415 bytes)


In [7]:
# ── Load and Clean World Bank Pink Sheet ────────────────────────────────────
# Actual column names confirmed from the downloaded file.
# Date format in index: '1960M01' (YYYYMmm) — not Jan-60.
# Note: cobalt, manganese, lithium not in Pink Sheet — covered by BGS production data.

METAL_COLS = {
    'Copper':             'copper',
    'Nickel':             'nickel',
    'Gold':               'gold',
    'Silver':             'silver',
    'Aluminum':           'aluminum',
    'Tin':                'tin',
    'Lead':               'lead',
    'Zinc':               'zinc',
    'Iron ore, cfr spot': 'iron_ore',
    'Platinum':           'platinum',
}


def load_pink_sheet(path: Path) -> pd.DataFrame:
    """
    Load World Bank Pink Sheet Excel and return tidy long-format prices.
    Handles index date format '1960M01' (YYYYMmm).

    Output schema: date, commodity, price_usd_mt, price_vol_12m

    Args:
        path: Path to pink_sheet.xlsx

    Returns:
        Long-format DataFrame filtered to START_YEAR–END_YEAR.
        Empty DataFrame if file missing or unparseable.

    Usage:
        prices_df = load_pink_sheet(WB_PATH)
    """
    if not path.exists():
        log.warning(f'Pink Sheet not found at {path}.')
        return pd.DataFrame()

    try:
        raw = pd.read_excel(path, sheet_name='Monthly Prices', header=4, index_col=0)
    except Exception as e:
        log.error(f'Could not parse Pink Sheet: {e}')
        return pd.DataFrame()

    # Parse '1960M01' format → datetime
    raw.index = pd.to_datetime(
        raw.index.astype(str).str.replace('M', '-', regex=False),
        format='%Y-%m',
        errors='coerce'
    )
    raw = raw[raw.index.notna()].sort_index()

    available = {k: v for k, v in METAL_COLS.items() if k in raw.columns}
    if not available:
        log.warning(f'No expected metal columns found. First 20 cols: {list(raw.columns[:20])}')
        return pd.DataFrame()

    prices = (
        raw[list(available.keys())]
        .rename(columns=available)
        .reset_index()
        .rename(columns={'index': 'date'})
        .melt(id_vars='date', var_name='commodity', value_name='price_usd_mt')
    )
    prices['price_usd_mt'] = pd.to_numeric(prices['price_usd_mt'], errors='coerce')
    prices = prices[
        prices['date'].dt.year.between(START_YEAR, END_YEAR)
    ].dropna(subset=['price_usd_mt'])

    log.info(f'Pink Sheet loaded: {prices.shape} | commodities: {sorted(prices["commodity"].unique())}')
    return prices


prices_df = load_pink_sheet(WB_PATH)

if not prices_df.empty:
    prices_df = prices_df.sort_values(['commodity', 'date'])
    prices_df['price_vol_12m'] = (
        prices_df.groupby('commodity')['price_usd_mt']
        .transform(lambda x: x.rolling(12, min_periods=6).std())
    )
    log.info('12-month rolling volatility computed.')
    display(prices_df.groupby('commodity')[['price_usd_mt', 'price_vol_12m']].describe().round(2))
else:
    log.warning('prices_df empty — check Pink Sheet download.')

18:19:42 [INFO] Pink Sheet loaded: (1560, 3) | commodities: ['aluminum', 'copper', 'gold', 'iron_ore', 'lead', 'nickel', 'platinum', 'silver', 'tin', 'zinc']
18:19:42 [INFO] 12-month rolling volatility computed.


price_usd_mt                                                   \
                 count      mean      std       min       25%       50%   
commodity                                                                 
aluminum         156.0   2025.63   372.36   1459.93   1770.94   1947.24   
copper           156.0   7070.94  1445.96   4471.79   5941.13   6881.68   
gold             156.0   1449.22   248.13   1075.74   1244.92   1341.10   
iron_ore         156.0    108.92    40.59     40.50     72.89    108.01   
lead             156.0   2094.70   240.14   1618.35   1936.76   2077.67   
nickel           156.0  16244.10  5207.26   8298.50  12461.49  15675.53   
platinum         156.0   1186.05   314.53    753.86    924.20   1040.66   
silver           156.0     21.45     6.40     14.11     16.63     19.35   
tin              156.0  21959.93  6062.61  13808.08  18223.27  20632.41   
zinc             156.0   2424.00   543.93   1520.36   2009.36   2293.45   

                              price_vol_12m                            \
                75%       max         count     mean      std     min   
commodity                                                               
aluminum    2208.37   3498.37         151.0   150.83    84.61   43.39   
copper      7996.50  10230.89         151.0   524.55   289.32  133.25   
gold        1681.38   1968.63         151.0    76.31    37.17   30.72   
iron_ore     137.14    214.43         151.0    15.47     8.40    4.41   
lead        2266.10   2701.17         151.0   149.56    62.66   36.12   
nickel     19123.56  33924.18         151.0  1909.18  1043.89  656.33   
platinum    1460.32   1825.90         151.0    78.53    31.48   25.90   
silver        25.32     42.70         151.0     2.22     1.71    0.38   
tin        22995.15  43983.35         151.0  2383.97  1945.64  397.99   
zinc        2752.60   4360.43         151.0   216.26    93.51   74.75   

                                               
               25%      50%      75%      max  
commodity                                      
aluminum    101.62   124.14   170.95   442.36  
copper      314.37   449.98   624.45  1360.33  
gold         48.45    60.75    99.43   172.46  
iron_ore     10.04    14.80    17.79    40.59  
lead        107.94   145.68   175.07   339.42  
nickel     1246.26  1727.90  2220.53  5651.53  
platinum     57.01    78.33   100.29   152.35  
silver        1.02     1.57     3.13     8.17  
tin        1009.78  1742.86  3137.51  9834.86  
zinc        141.42   211.75   270.36   449.75

## 1C — OECD Export Restrictions

**Source:** https://qdd.oecd.org/subject.aspx?Subject=ExportRestrictions_IndustrialRawMaterials  
**Manual step:** Go to the link above → export data → save as `data/raw/oecd/export_restrictions.csv` or `.xlsx`

Alternatively download from:  
https://www.oecd.org/en/publications/oecd-inventory-of-export-restrictions-on-industrial-raw-materials-2025_facc714b-en.html

In [8]:
# ── OECD Export Restrictions ────────────────────────────────────────────────
# File: 513026-oecd-industrial-raw-materials-complete-data.xlsx
# Location: data/raw/ST1/oecd/
# Close the file in Excel before running — lock file (~$...) causes read errors.

oecd_candidates = list(ST1_OECD.glob('*.csv')) + list(ST1_OECD.glob('*.xlsx'))


def load_oecd_restrictions(path: Path) -> pd.DataFrame:
    """
    Load OECD Export Restrictions Excel or CSV.
    Handles two-header-row structure: row 0 = full names, row 1 = short codes (skipped).
    Filters to critical minerals, keeps key policy columns.

    Args:
        path: Path to OECD export restrictions file

    Returns:
        Cleaned DataFrame with columns: iso3, country, hs_description, hs6,
        measure_type, year, date_introduced, direction.
        Empty DataFrame if file missing or unreadable.

    Usage:
        oecd_df = load_oecd_restrictions(ST1_OECD / '513026-oecd-industrial-raw-materials-complete-data.xlsx')
    """
    if not path.exists():
        log.warning(f'OECD file not found at {path}.')
        return pd.DataFrame()

    try:
        if path.suffix == '.xlsx':
            df = pd.read_excel(path, sheet_name='IND', header=0, skiprows=[1])
        else:
            df = pd.read_csv(path, encoding='utf-8-sig')
    except Exception as e:
        log.error(f'Could not read OECD file: {e}')
        return pd.DataFrame()

    log.info(f'OECD raw: {df.shape} | columns: {list(df.columns[:8])} ...')

    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
        .str.replace('/', '_', regex=False)
    )

    col_renames = {}
    for c in df.columns:
        if 'iso3' in c or c == 'cou':                    col_renames[c] = 'iso3'
        elif 'country_name' in c or c == 'country':      col_renames[c] = 'country'
        elif 'hs_desc' in c or 'hs_description' in c:    col_renames[c] = 'hs_description'
        elif c == 'hs6' or c.startswith('hs6'):          col_renames[c] = 'hs6'
        elif 'type_of' in c or c == 'type':              col_renames[c] = 'measure_type'
        elif c == 'year':                                 col_renames[c] = 'year'
        elif 'date_of_intro' in c or c == 'dati':        col_renames[c] = 'date_introduced'
        elif 'direction' in c or c == 'dir':              col_renames[c] = 'direction'

    df = df.rename(columns=col_renames)

    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    our_minerals = [
        'lithium', 'cobalt', 'gallium', 'germanium', 'rare earth',
        'graphite', 'nickel', 'copper', 'manganese', 'gold', 'silver'
    ]
    pattern = '|'.join(our_minerals)
    if 'hs_description' in df.columns:
        df = df[df['hs_description'].astype(str).str.lower().str.contains(pattern, na=False)]

    keep = [c for c in ['iso3', 'country', 'hs_description', 'hs6',
                         'measure_type', 'year', 'date_introduced', 'direction']
            if c in df.columns]
    df = df[keep].copy()
    df = df.dropna(subset=['iso3', 'year'])

    log.info(f'OECD filtered: {df.shape} | countries: {df["iso3"].nunique()} | '
             f'years: {df["year"].min()}–{df["year"].max()}')
    return df


if oecd_candidates:
    oecd_df = load_oecd_restrictions(oecd_candidates[0])
    if not oecd_df.empty:
        display(oecd_df.head(10))
        log.info(f'Measure types:\n{oecd_df["measure_type"].value_counts().to_string()}')
else:
    log.warning(f'No OECD file found in {ST1_OECD}.')
    oecd_df = pd.DataFrame()

18:19:42 [WARNING] No OECD file found in /Users/taruntheegela/Desktop/lambda/data/raw/ST1/oecd.


In [9]:
# ── Save All ST1 Outputs to Parquet ────────────────────────────────────────
# All ST1 processed files go to data/processed/ST1/

if not prod_df.empty:
    save_parquet(prod_df, ST1_PROC / 'st1_minerals_production.parquet', 'ST1 Production')
else:
    log.warning('prod_df empty — st1_minerals_production.parquet not saved.')

if not hhi_df.empty:
    save_parquet(hhi_df, ST1_PROC / 'st1_hhi.parquet', 'ST1 HHI')
else:
    log.warning('hhi_df empty — st1_hhi.parquet not saved.')

if not prices_df.empty:
    save_parquet(prices_df, ST1_PROC / 'st1_prices.parquet', 'ST1 Prices')
else:
    log.warning('prices_df empty — st1_prices.parquet not saved.')

if not oecd_df.empty:
    save_parquet(oecd_df, ST1_PROC / 'st1_oecd_restrictions.parquet', 'ST1 OECD Restrictions')
else:
    log.warning('oecd_df empty — st1_oecd_restrictions.parquet not saved.')

log.info('Phase 1 ST1 complete. Outputs in data/processed/ST1/')

18:19:42 [WARNING] prod_df empty — st1_minerals_production.parquet not saved.
18:19:42 [WARNING] hhi_df empty — st1_hhi.parquet not saved.
18:19:42 [INFO] Saved ST1 Prices: 1,560 rows × 4 cols → /Users/taruntheegela/Desktop/lambda/data/processed/ST1/st1_prices.parquet
18:19:42 [WARNING] oecd_df empty — st1_oecd_restrictions.parquet not saved.
18:19:42 [INFO] Phase 1 ST1 complete. Outputs in data/processed/ST1/
